# Build a gp_collab_hazel bundle from the Doyle rxnpredict screen

Source: [doylelab/rxnpredict](https://github.com/doylelab/rxnpredict) — the Ahneman
et al. (*Science* 2018) Buchwald–Hartwig C–N coupling screen. This notebook
downloads it, rebuilds the conditions-only table and the published DFT
descriptor table, attaches Kraken ligand chemistry, and writes an input bundle
laid out exactly like `gp_collab_hazel/inputs`.

Nothing is fitted here and no training happens. Needs
`numpy pandas scipy scikit-learn`; no PyTorch.

Run it from `gp_collab_hazel/prep_rxnpredict/`, with `gpc/` and `inputs/` in the
parent folder.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))
import rxnprep as R
from gpc.features import ReferencePCA

RAW = Path("raw_rxnpredict")          # download cache
PERERA = Path("../inputs")            # the existing Perera bundle, for the frozen PCA
OUT = Path("../inputs_rxnpredict")    # what this notebook writes
REFRESH = False                       # True re-downloads from GitHub

pd.set_option("display.width", 200, "display.max_columns", 60)
sources = R.download_sources(RAW, refresh=REFRESH)
display(pd.DataFrame(sources).T[["bytes", "sha256"]])

downloaded data_table.csv
downloaded ligand.csv
downloaded base.csv
downloaded aryl_halide.csv
downloaded additive.csv
downloaded ligand-list.csv
downloaded output_table.csv
downloaded kraken_identifiers.csv


,bytes,sha256
data_table.csv,1246868,fe310b50897e97578078558909efc2edaa7b98ad8939b1...
ligand.csv,3348,1b96791f1378593df1f2bb0c425a4bf06388b577760a04...
base.csv,418,3b3bac99c4dda4a7ee181f7dd951798b1f67f724fcf945...
aryl_halide.csv,3864,0f979fd7e875f883b2998fb5e23abcb468bb6cef4511c2...
additive.csv,4006,3d9ba693e77b5087a827662d9412cb0c1179eadb2df848...
ligand-list.csv,426,19c78fede9a3dd3ee8750561a151fe6248aa03d764ecb1...
output_table.csv,3201887,ffbfa7de0ec175ae83a24fa40e1bbbe9c23fbcae8b3a48...
kraken_identifiers.csv,245844,6d59540fe2a55326d257f9d7d750d058dfcb32e4806e86...


## 1. Is the descriptor join correct?

`R/output_table.csv` has no key columns and Doyle's R script attaches yields by
row position, which is fragile. Instead the descriptors are joined on component
name. To show that join is right, the full factorial grid is rebuilt from the
four per-component tables and compared against the published table.

`identical_row_multiset: True` means the two agree exactly, ignoring row order.

In [2]:
check = R.verify_against_doyle(RAW)
display(pd.Series(check).to_frame("result"))
assert check["same_columns"] and check["identical_row_multiset"], "join does not reproduce Doyle's table"

,result
published_shape,"[3960, 120]"
rebuilt_shape,"[3960, 120]"
same_columns,True
identical_row_multiset,True


## 2. The conditions-only dataframe

Reaction conditions and yield, nothing else — no NMR shifts, charges,
vibrations or volumes. Two groups of rows are dropped, both reported in the
audit below:

- **no-aryl-halide control wells**, which are blank by design (every one is 0% yield)
- **one additive with no computed descriptors** (`5-Phenyl-1,2,4-oxadiazole`), plus wells with no additive

What remains is the 3,955-reaction set: 4 ligands × 3 bases × 15 aryl halides ×
22 additives, minus 5 combinations that were never run.

In [3]:
reactions, audit = R.build_reactions(RAW)
print(f"conditions-only dataframe: {reactions.shape[0]} reactions x {reactions.shape[1]} columns")
display(reactions.head())

print("\nDropped:")
display(pd.DataFrame([
    {"reason": "no-aryl-halide controls", "rows": audit["dropped_no_aryl_halide_controls"]["rows"],
     "note": f"max yield {audit['dropped_no_aryl_halide_controls']['max_yield']}"},
    {"reason": "additive without descriptors", "rows": audit["dropped_additive_without_descriptors"]["rows"],
     "note": ", ".join(audit["dropped_additive_without_descriptors"]["names"])},
]))
print(f"kept {audit['kept_rows']} of a {audit['full_factorial']}-cell grid; "
      f"{len(audit['missing_combinations'])} combinations were never run")
display(pd.DataFrame(audit["missing_combinations"], columns=["ligand","base","aryl_halide","additive"]))
print("\nRows per ligand (these are the LOLO fold sizes):")
display(pd.Series(audit["rows_per_ligand"], name="reactions").to_frame())

conditions-only dataframe: 3955 reactions x 10 columns


,row_id,ligand,base,aryl_halide,additive,yield,plate,row,col,kraken_id
0,reaction:1,XPhos,P2Et,1-chloro-4-(trifluoromethyl)benzene,5-phenylisoxazole,10.657812,1,2,1,1
1,reaction:2,XPhos,P2Et,1-bromo-4-(trifluoromethyl)benzene,5-phenylisoxazole,14.747896,1,2,2,1
2,reaction:3,XPhos,P2Et,1-iodo-4-(trifluoromethyl)benzene,5-phenylisoxazole,18.278686,1,2,3,1
3,reaction:4,XPhos,P2Et,1-chloro-4-methoxybenzene,5-phenylisoxazole,2.475058,1,2,4,1
4,reaction:5,XPhos,P2Et,1-bromo-4-methoxybenzene,5-phenylisoxazole,6.119058,1,2,5,1



Dropped:


,reason,rows,note
0,no-aryl-halide controls,263,max yield 0.0
1,additive without descriptors,381,"5-Phenyl-1,2,4-oxadiazole"


kept 3955 of a 3960-cell grid; 5 combinations were never run


,ligand,base,aryl_halide,additive
0,t-BuBrettPhos,BTMG,3-iodopyridine,3-phenylisoxazole
1,t-BuXPhos,BTMG,1-chloro-4-ethylbenzene,3-phenylisoxazole
2,t-BuXPhos,BTMG,1-chloro-4-ethylbenzene,4-phenylisoxazole
3,t-BuXPhos,MTBD,1-bromo-4-(trifluoromethyl)benzene,3-methylisoxazole
4,t-BuXPhos,MTBD,1-iodo-4-(trifluoromethyl)benzene,3-methylisoxazole



Rows per ligand (these are the LOLO fold sizes):


,reactions
AdBrettPhos,990
XPhos,990
t-BuBrettPhos,989
t-BuXPhos,986


## 3. The published descriptor dataframe

The same 120 DFT descriptors Doyle modelled — 64 ligand, 27 aryl halide, 19
additive, 10 base — one row per reaction, keyed by `row_id`. This is the
`rxnpredict_full` baseline.

In [4]:
rxn_features = R.build_rxnpredict_features(reactions, RAW)
print(f"{rxn_features.shape[0]} reactions x {rxn_features.shape[1]-1} descriptors")
counts = {}
for c in rxn_features.columns[1:]:
    counts[c.split("_")[0]] = counts.get(c.split("_")[0], 0) + 1
display(pd.Series(counts, name="descriptors").to_frame())
display(rxn_features.iloc[:4, :6])

constant = [c for c in rxn_features.columns[1:] if rxn_features[c].std() == 0]
print(f"constant (zero-variance) descriptors: {len(constant)}")

3955 reactions x 120 descriptors


,descriptors
ligand,64
base,10
aryl,27
additive,19


,row_id,ligand_*C10_NMR_shift,ligand_*C10_electrostatic_charge,ligand_*C11_NMR_shift,ligand_*C11_electrostatic_charge,ligand_*C12_NMR_shift
0,reaction:1,139.58,0.124,114.03,-0.248,140.09
1,reaction:2,139.58,0.124,114.03,-0.248,140.09
2,reaction:3,139.58,0.124,114.03,-0.248,140.09
3,reaction:4,139.58,0.124,114.03,-0.248,140.09


constant (zero-variance) descriptors: 0


## 4. Kraken ligands

All four Doyle ligands exist in the Kraken reference, so the same ligand
chemistry used for Perera is available here. Check the two SMILES columns agree
chemically before trusting the mapping.

In [5]:
display(R.verify_ligand_map(RAW))

ligands = pd.read_csv(PERERA / "ligand_features.csv")
reference = ReferencePCA.load(PERERA)
used = ligands[ligands.kraken_id.isin(R.LIGAND_KRAKEN.values())]
print(f"frozen Kraken PCA: {len(reference.columns)} descriptors, "
      f"{reference.variance_ratio.shape[0]} components, "
      f"explained variance {(reference.variance_ratio*100).round(2).tolist()}%")
print("top 3 descriptors per component:", reference.top_by_pc)
display(used[["kraken_id"] + R.FIVE + ["PC1","PC2","PC3","PC4"]].round(3))

,ligand,kraken_id,kraken_name,rxnpredict_smiles,kraken_smiles
0,XPhos,1,Xphos,CC(C)C1=CC(C(C)C)=CC(C(C)C)=C1C2=C(P(C3CCCCC3)...,CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2...
1,t-BuXPhos,90,tBuXPhos,CC(C)C(C=C(C(C)C)C=C1C(C)C)=C1C2=CC=CC=C2P(C(C...,CC(C)c1cc(C(C)C)c(-c2ccccc2P(C(C)(C)C)C(C)(C)C...
2,t-BuBrettPhos,89,tBuBrettPhos,CC(C)C1=CC(C(C)C)=CC(C(C)C)=C1C2=C(P(C(C)(C)C)...,COc1ccc(OC)c(P(C(C)(C)C)C(C)(C)C)c1-c1c(C(C)C)...
3,AdBrettPhos,347,AdBrettPhos,CC(C1=C(C2=C(OC)C=CC(OC)=C2P(C34CC5CC(C4)CC(C5...,COc1ccc(c(c1c1c(cc(cc1C(C)C)C(C)C)C(C)C)P(C12C...


frozen Kraken PCA: 190 descriptors, 4 components, explained variance [29.0, 13.01, 11.18, 6.15]%
top 3 descriptors per component: {'PC1': ['vbur_vtot_boltz', 'volume_boltz', 'vbur_near_vtot_vburminconf'], 'PC2': ['pyr_P_max', 'qpole_amp_max', 'vbur_qvbur_min_min'], 'PC3': ['vbur_near_vbur_delta', 'vbur_vbur_delta', 'vbur_qvbur_min_delta'], 'PC4': ['nbo_bds_occ_avg_boltz', 'efgtens_zz_P_boltz', 'nmr_P_boltz']}


,kraken_id,vbur_pct_boltz,vbur_pct_min,vbur_pct_delta,dipolemoment_boltz,homo_lumo_gap_eV,PC1,PC2,PC3,PC4
0,1,58.780,31.436,39.797,1.258,5.255,11.147,-0.575,7.390,-4.352
1,89,67.755,43.926,27.834,3.631,4.842,13.050,-6.004,2.503,2.825
2,90,63.084,56.728,10.657,1.119,5.120,9.311,-11.243,1.773,8.074
3,347,67.168,41.432,29.517,3.562,4.798,16.399,-5.630,1.909,1.472


## 5. The seven feature sets

Five carry ligand chemistry and share the same 40-column one-hot block
(15 aryl halides + 3 bases + 22 additives). Two are the new rxnpredict
baselines: `rxnpredict_full` is the published model's descriptors with **no**
one-hot block at all, and `rxnpredict_full_ohe` adds the shared block so the
only thing changing across the other six is the ligand representation.

In [6]:
cfg = R.default_config()
display(R.model_table(reactions, ligands, rxn_features, cfg, reference))

for model in cfg["features"]["models"]:
    frame = R.feature_frame(reactions, ligands, rxn_features, model, cfg, reference)
    print(f"  {model:22s} raw columns {frame.shape[1]:4d}  ->  ok")

,model,ligand_features,ligand_descriptors,rxnpredict_descriptors,numeric_inputs,onehot_inputs,encoded_inputs,categorical_fields
0,ligand_ohe,Ligand identity one-hot,0,0,0,44,44,"ligand, aryl_halide, base, additive"
1,selected_2,Boltzmann-average and minimum buried volume,2,0,2,40,42,"aryl_halide, base, additive"
2,selected_5,"Buried volume boltz/min/range, dipole, HOMO-LU...",5,0,5,40,45,"aryl_halide, base, additive"
3,pc_top,"Top 3 Kraken descriptors from each of PC1-PC4,...",12,0,12,40,52,"aryl_halide, base, additive"
4,pc_scores,Every Kraken descriptor times its loading in e...,190,0,760,40,800,"aryl_halide, base, additive"
5,rxnpredict_full,Published DFT descriptors for all four compone...,0,120,120,0,120,(none)
6,rxnpredict_full_ohe,The same DFT descriptors plus the shared one-h...,0,120,120,40,160,"aryl_halide, base, additive"


  ligand_ohe             raw columns    4  ->  ok
  selected_2             raw columns    5  ->  ok
  selected_5             raw columns    8  ->  ok
  pc_top                 raw columns   15  ->  ok
  pc_scores              raw columns  193  ->  ok
  rxnpredict_full        raw columns  120  ->  ok
  rxnpredict_full_ohe    raw columns  123  ->  ok


### Every encoded input, per set

`model_features.csv` in the bundle records all of them; this shows the head of each.

In [8]:
features = R.model_features(reactions, ligands, rxn_features, cfg, reference)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(features[["model", "feature"]])

,model,feature
0,ligand_ohe,cat__ligand_AdBrettPhos
1,ligand_ohe,cat__ligand_XPhos
2,ligand_ohe,cat__ligand_t-BuBrettPhos
3,ligand_ohe,cat__ligand_t-BuXPhos
4,ligand_ohe,cat__aryl_halide_1-bromo-4-(trifluoromethyl)benzene
5,ligand_ohe,cat__aryl_halide_1-bromo-4-ethylbenzene
6,ligand_ohe,cat__aryl_halide_1-bromo-4-methoxybenzene
7,ligand_ohe,cat__aryl_halide_1-chloro-4-(trifluoromethyl)benzene
8,ligand_ohe,cat__aryl_halide_1-chloro-4-ethylbenzene
9,ligand_ohe,cat__aryl_halide_1-chloro-4-methoxybenzene


## 6. Evaluation plan

With 4 ligands, LOLO is 4 folds and each holds out about a quarter of the data
(it was 8 folds and an eighth for Perera). The stratified 4-fold keeps LOLO's
fold geometry with every ligand present on both sides, so it stays the matched
in-distribution control; the stratified 5-fold is the ordinary CV. Stratification
is on ligand identity, seed 42, and there is no 80/20 holdout.

In [9]:
sys.path.insert(0, str(Path.cwd().parent))
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
import numpy as np

groups = reactions[R.GROUP].to_numpy()
idx = np.arange(len(reactions))
plan = []
for name, folds in (
        ("lolo", list(LeaveOneGroupOut().split(idx, groups=groups))),
        ("iid_stratified_4", list(StratifiedKFold(4, shuffle=True, random_state=42).split(idx, groups))),
        ("kfold_stratified_5", list(StratifiedKFold(5, shuffle=True, random_state=42).split(idx, groups)))):
    for i, (tr, te) in enumerate(folds):
        plan.append({"method": name, "fold": i, "n_train": len(tr), "n_test": len(te),
                     "test_ligands": len(set(groups[te])),
                     "held_out": sorted(set(groups[te]))[0] if name == "lolo" else ""})
plan = pd.DataFrame(plan)
display(plan)
print(f"{len(plan)} folds x {len(cfg['features']['models'])} feature sets = "
      f"{len(plan)*len(cfg['features']['models'])} GP fits")

,method,fold,n_train,n_test,test_ligands,held_out
0,lolo,0,2965,990,1,AdBrettPhos
1,lolo,1,2965,990,1,XPhos
2,lolo,2,2966,989,1,t-BuBrettPhos
3,lolo,3,2969,986,1,t-BuXPhos
4,iid_stratified_4,0,2966,989,4,
5,iid_stratified_4,1,2966,989,4,
6,iid_stratified_4,2,2966,989,4,
7,iid_stratified_4,3,2967,988,4,
8,kfold_stratified_5,0,3164,791,4,
9,kfold_stratified_5,1,3164,791,4,


13 folds x 7 feature sets = 91 GP fits


## 7. Export the bundle

Writes `../inputs_rxnpredict/` with the same file layout as `inputs/`. The
frozen PCA files are **copied, never refitted**, so `pc_top` and `pc_scores`
mean exactly what they mean in the Perera bundle.

In [10]:
out = R.export_bundle(OUT, reactions, ligands, rxn_features, cfg, reference,
                     audit, sources, PERERA)
print("wrote", out.resolve())
display(pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size/1024, 1)}
                      for p in sorted(out.iterdir())]))

import json
manifest = json.loads((out / "manifest.json").read_text())
print({k: manifest[k] for k in ("dataset", "n_rows", "n_models", "n_methods")})
print("bundle_id:", manifest["bundle_id"])

wrote C:\Users\kagoble\Documents\GitHub\DOPE-MURI\gp_collab_rxnpredict\inputs_rxnpredict


,file,KB
0,audit.json,2.9
1,config.json,1.1
2,ligand_features.csv,13.0
3,ligand_mapping.csv,0.1
4,manifest.json,1.3
5,model_features.csv,70.5
6,model_table.csv,0.9
7,pca_loadings.csv,18.9
8,pca_reference.json,5.8
9,pca_reference.npz,12.1


{'dataset': 'rxnpredict_ahneman_2018', 'n_rows': 3955, 'n_models': 7, 'n_methods': 4}
bundle_id: 52a3ccb5f174b4c432a3c97394ab4a78b808f32c148f578236675c3d2c1e36aa
